# 400 — Cross-System Program Comparison

## Objective

Compare the independently discovered and frozen candidate-program spaces from the TCGA tumor and DepMap–GDSC cell-line discovery layers to identify supported, partial, ambiguous, or non-recoverable cross-system transcriptomic correspondences.

The primary comparison is performed between the **10 unique tumor RNA ICA axes** represented among the 13 retained TCGA cross-omic candidate programs and the **10 frozen cell-line ICA candidate programs** identified in notebook 310.

This notebook will:

- harmonize the frozen tumor and cell-line gene-loading spaces;
- compare candidate programs using prespecified multiview transcriptomic matching;
- handle ICA sign indeterminacy without assuming portable component indices;
- evaluate correspondence using loading-based structural evidence and an empirical null;
- distinguish `SUPPORTED_CORRESPONDENCE`, `PARTIAL_CORRESPONDENCE`, `AMBIGUOUS_CORRESPONDENCE`, and `NOT_RECOVERABLE`;
- propagate transcriptomic correspondence results back to the 13 tumor cross-omic candidate arms;
- preserve tumor structural-family membership and upstream robustness, confounding, and technical-caution metadata;
- produce a frozen cross-system comparison layer for notebook 401.

## Scope

This notebook does **not** rediscover tumor or cell-line programs, redefine either candidate universe, construct consensus programs, perform biological annotation, or use pharmacological phenotype associations to select cross-system matches.

Tumor and cell-line discovery remain analytically independent.

Component indices are not assumed to be portable across systems. Cross-system correspondence is determined from quantitative representation-level evidence.

A `NOT_RECOVERABLE` result is considered a valid scientific outcome and indicates lack of sufficient correspondence within the frozen Phase 3 candidate space; it does not establish biological absence of the tumor program in cell models.

Cross-system correspondence provides computational evidence of reproducibility only when the prespecified support criteria are satisfied. It does not constitute causal, clinical, or independent biological validation.

## Methodological boundary

The primary statistical matching unit is the **unique tumor transcriptomic axis**, rather than the 13 tumor cross-omic pairs, because several tumor candidate arms share the same RNA component.

The 13 tumor cross-omic candidates remain distinct biological-computational outputs and will be restored after transcriptomic matching together with their methylation component, structural-family membership, relationship form, and upstream evidence status.

This design prevents shared tumor RNA axes from being counted multiple times as independent cross-system evidence.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import json

import numpy as np
import pandas as pd

from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

from pancancer_epigenetics.utils.paths import (
    Paths,
    project_relative_path,
)

In [2]:
# =============================================================================
# Input and output directories
# =============================================================================

TUMOR_PROGRAM_DIR = Paths.tumor_programs
CELL_LINE_PROGRAM_DIR = Paths.cellline_programs
OUTPUT_DIR = Paths.consensus_programs

In [3]:
# =============================================================================
# Authoritative program artifact paths
# =============================================================================

TUMOR_CANDIDATE_CATALOG_PATH = (
    TUMOR_PROGRAM_DIR
    / "tcga_primary_tumor_cross_omic_candidate_pair_catalog.csv"
)

TUMOR_RNA_LOADINGS_PATH = (
    TUMOR_PROGRAM_DIR
    / "tcga_primary_tumor_rna_ica_candidate_gene_loadings.csv"
)

CELL_LINE_ICA_LOADINGS_PATH = (
    CELL_LINE_PROGRAM_DIR
    / "310_ica_program_loadings.parquet"
)

CELL_LINE_ASSOCIATIONS_PATH = (
    CELL_LINE_PROGRAM_DIR
    / "310_program_phenotype_associations.csv"
)

CELL_LINE_ROBUSTNESS_PATH = (
    CELL_LINE_PROGRAM_DIR
    / "311_program_robustness_summary.csv"
)

In [4]:
# =============================================================================
# Load authoritative program artifacts
# =============================================================================

tumor_candidate_catalog = pd.read_csv(
    TUMOR_CANDIDATE_CATALOG_PATH
)

tumor_rna_loadings = pd.read_csv(
    TUMOR_RNA_LOADINGS_PATH
)

cell_line_ica_loadings = pd.read_parquet(
    CELL_LINE_ICA_LOADINGS_PATH
)

cell_line_associations = pd.read_csv(
    CELL_LINE_ASSOCIATIONS_PATH
)

cell_line_robustness = pd.read_csv(
    CELL_LINE_ROBUSTNESS_PATH
)

In [5]:
# =============================================================================
# Define frozen cross-system candidate universes
# =============================================================================

tumor_candidates = (
    tumor_candidate_catalog
    .loc[
        tumor_candidate_catalog["sex_sensitivity_status"]
        .eq("retained_after_sex_sensitivity")
    ]
    .copy()
)

cell_line_candidates = (
    cell_line_associations
    .loc[cell_line_associations["q_value"].lt(0.05)]
    .copy()
)

print("Tumor candidate arms :", len(tumor_candidates))
print("Cell-line candidates :", len(cell_line_candidates))

Tumor candidate arms : 13
Cell-line candidates : 10


In [6]:
# =============================================================================
# Define unique tumor transcriptomic axes
# =============================================================================

tumor_rna_axes = (
    tumor_candidates["rna_component"]
    .drop_duplicates()
    .tolist()
)

cell_line_program_ids = (
    cell_line_candidates["program_id"]
    .tolist()
)

print("Unique tumor RNA axes :", len(tumor_rna_axes))
print("Cell-line programs    :", len(cell_line_program_ids))
print("Primary comparisons   :", len(tumor_rna_axes) * len(cell_line_program_ids))

Unique tumor RNA axes : 10
Cell-line programs    : 10
Primary comparisons   : 100


In [7]:
# =============================================================================
# Inspect loading-table structure for gene harmonization
# =============================================================================

print("Tumor RNA loadings columns:")
print(tumor_rna_loadings.columns.tolist())

print("\nCell-line ICA loadings columns:")
print(cell_line_ica_loadings.columns.tolist())

Tumor RNA loadings columns:
['matrix_row_index', 'gene_id', 'gene_name', 'gene_type', 'gene_id_base', 'gene_id_is_versioned', 'expressed_sample_count', 'filtered_matrix_row_index', 'mean_logcpm', 'variance_logcpm', 'median_within_project_variance', 'RNA_IC158', 'RNA_IC083', 'RNA_IC184', 'RNA_IC150', 'RNA_IC169', 'RNA_IC175', 'RNA_IC050', 'RNA_IC151', 'RNA_IC001', 'RNA_IC193']

Cell-line ICA loadings columns:
['gene', 'ICA_PROGRAM_01', 'ICA_PROGRAM_02', 'ICA_PROGRAM_03', 'ICA_PROGRAM_04', 'ICA_PROGRAM_05', 'ICA_PROGRAM_06', 'ICA_PROGRAM_07', 'ICA_PROGRAM_08', 'ICA_PROGRAM_09', 'ICA_PROGRAM_10', 'ICA_PROGRAM_11', 'ICA_PROGRAM_12', 'ICA_PROGRAM_13', 'ICA_PROGRAM_14', 'ICA_PROGRAM_15', 'ICA_PROGRAM_16', 'ICA_PROGRAM_17', 'ICA_PROGRAM_18', 'ICA_PROGRAM_19', 'ICA_PROGRAM_20', 'ICA_PROGRAM_21', 'ICA_PROGRAM_22', 'ICA_PROGRAM_23', 'ICA_PROGRAM_24', 'ICA_PROGRAM_25', 'ICA_PROGRAM_26', 'ICA_PROGRAM_27', 'ICA_PROGRAM_28', 'ICA_PROGRAM_29', 'ICA_PROGRAM_30', 'ICA_PROGRAM_31', 'ICA_PROGRAM_32', 'IC

In [8]:
# =============================================================================
# Inspect gene identifier formats
# =============================================================================

print("Tumor gene identifiers:")
print(tumor_rna_loadings["gene_name"].head(10).tolist())

print("\nCell-line gene identifiers:")
print(cell_line_ica_loadings["gene"].head(10).tolist())

Tumor gene identifiers:
['TNMD', 'CFH', 'CFTR', 'CYP51A1', 'HS3ST1', 'AOC1', 'WNT16', 'HECW1', 'TMEM176A', 'KLHL13']

Cell-line gene identifiers:
['KRT19 (3880)', 'SPARC (6678)', 'C19orf33 (64073)', 'KRT7 (3855)', 'VIM (7431)', 'UCHL1 (7345)', 'RPS4Y1 (6192)', 'TGFBI (7045)', 'KRT8 (3856)', 'S100P (6286)']


In [9]:
# =============================================================================
# Derive harmonized gene symbols
# =============================================================================

tumor_rna_loadings_harmonized = tumor_rna_loadings.assign(
    gene_symbol=tumor_rna_loadings["gene_name"]
)

cell_line_ica_loadings_harmonized = cell_line_ica_loadings.assign(
    gene_symbol=cell_line_ica_loadings["gene"].str.replace(
        r"\s+\(\d+\)$",
        "",
        regex=True,
    )
)

In [10]:
# =============================================================================
# Check harmonized gene-symbol uniqueness and overlap
# =============================================================================

tumor_gene_duplicates = (
    tumor_rna_loadings_harmonized["gene_symbol"]
    .duplicated()
    .sum()
)

cell_line_gene_duplicates = (
    cell_line_ica_loadings_harmonized["gene_symbol"]
    .duplicated()
    .sum()
)

shared_gene_symbols = sorted(
    set(tumor_rna_loadings_harmonized["gene_symbol"])
    & set(cell_line_ica_loadings_harmonized["gene_symbol"])
)

print("Tumor duplicated gene symbols    :", tumor_gene_duplicates)
print("Cell-line duplicated gene symbols:", cell_line_gene_duplicates)
print("Shared gene symbols               :", len(shared_gene_symbols))

Tumor duplicated gene symbols    : 0
Cell-line duplicated gene symbols: 0
Shared gene symbols               : 2389


In [11]:
# =============================================================================
# Build shared cross-system gene universe
# =============================================================================

shared_gene_universe = (
    tumor_rna_loadings_harmonized[
        ["gene_id", "gene_name", "gene_symbol"]
    ]
    .merge(
        cell_line_ica_loadings_harmonized[
            ["gene", "gene_symbol"]
        ],
        on="gene_symbol",
        how="inner",
    )
    .rename(
        columns={
            "gene_id": "tumor_gene_id",
            "gene_name": "tumor_gene_name",
            "gene": "cell_line_gene_id",
        }
    )
    .sort_values("gene_symbol")
    .reset_index(drop=True)
)

In [12]:
# =============================================================================
# Summarize shared gene-space coverage
# =============================================================================

shared_gene_count = len(shared_gene_universe)

tumor_gene_count = len(tumor_rna_loadings_harmonized)
cell_line_gene_count = len(cell_line_ica_loadings_harmonized)

print("Tumor RNA genes          :", tumor_gene_count)
print("Cell-line ICA genes      :", cell_line_gene_count)
print("Shared genes             :", shared_gene_count)
print(
    "Tumor-space coverage     :",
    f"{shared_gene_count / tumor_gene_count:.2%}",
)
print(
    "Cell-line-space coverage :",
    f"{shared_gene_count / cell_line_gene_count:.2%}",
)

Tumor RNA genes          : 5000
Cell-line ICA genes      : 5000
Shared genes             : 2389
Tumor-space coverage     : 47.78%
Cell-line-space coverage : 47.78%


In [13]:
# =============================================================================
# Build shared-gene loading matrices
# =============================================================================

tumor_shared_loadings = (
    shared_gene_universe[["gene_symbol"]]
    .merge(
        tumor_rna_loadings_harmonized[
            ["gene_symbol", *tumor_rna_axes]
        ],
        on="gene_symbol",
        how="left",
    )
    .set_index("gene_symbol")
)

cell_line_shared_loadings = (
    shared_gene_universe[["gene_symbol"]]
    .merge(
        cell_line_ica_loadings_harmonized[
            ["gene_symbol", *cell_line_program_ids]
        ],
        on="gene_symbol",
        how="left",
    )
    .set_index("gene_symbol")
)

In [14]:
# =============================================================================
# Confirm shared loading matrices
# =============================================================================

print("Tumor shared loadings shape    :", tumor_shared_loadings.shape)
print("Cell-line shared loadings shape:", cell_line_shared_loadings.shape)

print(
    "Tumor missing loadings         :",
    int(tumor_shared_loadings.isna().sum().sum()),
)
print(
    "Cell-line missing loadings     :",
    int(cell_line_shared_loadings.isna().sum().sum()),
)

Tumor shared loadings shape    : (2389, 10)
Cell-line shared loadings shape: (2389, 10)
Tumor missing loadings         : 0
Cell-line missing loadings     : 0


## Prespecified cross-system matching policy

Cross-system correspondence is evaluated exclusively within the frozen candidate spaces defined above.

The primary statistical unit is the unique TCGA RNA axis. Tumor cross-omic arms sharing the same RNA component are therefore not treated as independent cross-system comparisons.

### Structural evidence

For every tumor RNA axis–cell-line ICA candidate pair:

1. **Pearson loading correlation** across the frozen shared-gene universe is the primary structural similarity measure.
2. **Absolute Pearson correlation** is used for matching because ICA component signs are arbitrary.
3. The sign of the Pearson correlation defines the cell-line orientation relative to the tumor axis.
4. **Spearman loading correlation** is used as a secondary rank-based concordance measure.
5. Signed high-loading overlap using the top 10% of absolute loadings is used as an additional secondary structural view.
6. A top-5% loading overlap may be evaluated only as a prespecified sensitivity analysis.

Pharmacological phenotype associations, tumor robustness status, structural-family membership, methylation behavior, and biological annotation are not used to select, orient, or threshold cross-system matches.

### Statistical assessment

Primary Pearson correspondence will be evaluated against an empirical gene-label permutation null using 10,000 permutations with a fixed random seed.

Multiple-testing correction will be performed across the 100 frozen primary tumor–cell-line comparisons using Benjamini–Hochberg FDR.

### Correspondence classes

`SUPPORTED_CORRESPONDENCE` requires:

- empirical Pearson FDR < 0.05;
- reciprocal nearest-neighbor correspondence between the frozen tumor and cell-line candidate spaces;
- support from at least one secondary structural view.

`PARTIAL_CORRESPONDENCE` denotes statistically supported structural similarity without sufficient reciprocal or multiview evidence for supported correspondence.

`AMBIGUOUS_CORRESPONDENCE` denotes unresolved competition among multiple plausible candidate matches or disagreement among structural views.

`NOT_RECOVERABLE` denotes absence of sufficient correspondence within the frozen Phase 3 candidate space.

No correspondence class may be upgraded using phenotype association strength or upstream biological interpretability.

A transcriptomic correspondence does not establish reproduction of the complete tumor epigenetic-transcriptomic program in cell lines.

In [15]:
# =============================================================================
# Prespecified cross-system matching parameters
# =============================================================================

RANDOM_SEED = 20260818
N_PERMUTATIONS = 10_000
FDR_THRESHOLD = 0.05

TOP_LOADING_FRACTION = 0.10
TOP_LOADING_SENSITIVITY_FRACTION = 0.05

In [16]:
# =============================================================================
# Compute observed primary loading correlations
# =============================================================================

primary_correlations = pd.DataFrame(
    [
        {
            "tumor_rna_axis": tumor_axis,
            "cell_line_program": cell_program,
            "pearson_r": np.corrcoef(
                tumor_shared_loadings[tumor_axis],
                cell_line_shared_loadings[cell_program],
            )[0, 1],
        }
        for tumor_axis in tumor_rna_axes
        for cell_program in cell_line_program_ids
    ]
)

# ICA signs are arbitrary across independent decompositions.
# Absolute correlation is used for matching; signed correlation defines orientation.
primary_correlations["abs_pearson_r"] = (
    primary_correlations["pearson_r"].abs()
)

primary_correlations["orientation_multiplier"] = np.sign(
    primary_correlations["pearson_r"]
).astype(int)

In [17]:
# =============================================================================
# Compute secondary rank-based loading correlations
# =============================================================================

spearman_correlations = pd.DataFrame(
    [
        {
            "tumor_rna_axis": tumor_axis,
            "cell_line_program": cell_program,
            "spearman_r": spearmanr(
                tumor_shared_loadings[tumor_axis],
                cell_line_shared_loadings[cell_program],
            ).statistic,
        }
        for tumor_axis in tumor_rna_axes
        for cell_program in cell_line_program_ids
    ]
)

# Align the secondary metric with the ICA orientation defined exclusively
# by the primary Pearson loading correlation.
primary_correlations = primary_correlations.merge(
    spearman_correlations,
    on=["tumor_rna_axis", "cell_line_program"],
    how="left",
)

primary_correlations["oriented_spearman_r"] = (
    primary_correlations["spearman_r"]
    * primary_correlations["orientation_multiplier"]
)

In [18]:
# =============================================================================
# Define signed high-loading overlap metric
# =============================================================================

def signed_top_loading_overlap(
    tumor_loadings,
    cell_loadings,
    orientation_multiplier,
    top_fraction,
):
    top_n = int(np.ceil(len(tumor_loadings) * top_fraction))

    tumor_top = tumor_loadings.abs().nlargest(top_n).index
    cell_top = cell_loadings.abs().nlargest(top_n).index
    shared_top = tumor_top.intersection(cell_top)

    # Orient the cell-line component using the primary Pearson sign before
    # evaluating whether shared extreme loadings point in the same direction.
    oriented_cell = cell_loadings * orientation_multiplier

    concordant_positive = (
        (tumor_loadings.loc[shared_top] > 0)
        & (oriented_cell.loc[shared_top] > 0)
    ).sum()

    concordant_negative = (
        (tumor_loadings.loc[shared_top] < 0)
        & (oriented_cell.loc[shared_top] < 0)
    ).sum()

    discordant = len(shared_top) - concordant_positive - concordant_negative

    return {
        "top_n": top_n,
        "shared_top_count": len(shared_top),
        "shared_top_fraction": len(shared_top) / top_n,
        "concordant_positive_count": concordant_positive,
        "concordant_negative_count": concordant_negative,
        "discordant_count": discordant,
    }

In [19]:
# =============================================================================
# Compute primary signed high-loading overlap
# =============================================================================

top_loading_overlap = pd.DataFrame(
    [
        {
            "tumor_rna_axis": row.tumor_rna_axis,
            "cell_line_program": row.cell_line_program,
            **signed_top_loading_overlap(
                tumor_shared_loadings[row.tumor_rna_axis],
                cell_line_shared_loadings[row.cell_line_program],
                row.orientation_multiplier,
                TOP_LOADING_FRACTION,
            ),
        }
        for row in primary_correlations.itertuples(index=False)
    ]
)

primary_correlations = primary_correlations.merge(
    top_loading_overlap,
    on=["tumor_rna_axis", "cell_line_program"],
    how="left",
)

In [20]:
# =============================================================================
# Prepare normalized loading matrices for permutation testing
# =============================================================================

tumor_loading_matrix = tumor_shared_loadings.to_numpy(dtype=float)
cell_line_loading_matrix = cell_line_shared_loadings.to_numpy(dtype=float)

# Center each program and scale it to unit Euclidean norm so that the
# cross-product between columns is exactly their Pearson correlation.
tumor_loading_matrix = (
    tumor_loading_matrix
    - tumor_loading_matrix.mean(axis=0, keepdims=True)
)
tumor_loading_matrix /= np.linalg.norm(
    tumor_loading_matrix,
    axis=0,
    keepdims=True,
)

cell_line_loading_matrix = (
    cell_line_loading_matrix
    - cell_line_loading_matrix.mean(axis=0, keepdims=True)
)
cell_line_loading_matrix /= np.linalg.norm(
    cell_line_loading_matrix,
    axis=0,
    keepdims=True,
)

In [21]:
# =============================================================================
# Compute empirical Pearson permutation p-values
# =============================================================================

observed_abs_correlations = (
    primary_correlations
    .pivot(
        index="tumor_rna_axis",
        columns="cell_line_program",
        values="abs_pearson_r",
    )
    .reindex(
        index=tumor_rna_axes,
        columns=cell_line_program_ids,
    )
    .to_numpy()
)

rng = np.random.default_rng(RANDOM_SEED)
exceedance_counts = np.zeros_like(
    observed_abs_correlations,
    dtype=int,
)

# A common gene-label permutation is applied to all 100 comparisons at each
# iteration, preserving the dependence structure within each program space.
for _ in range(N_PERMUTATIONS):
    permuted_indices = rng.permutation(shared_gene_count)

    permuted_correlations = (
        tumor_loading_matrix.T
        @ cell_line_loading_matrix[permuted_indices, :]
    )

    exceedance_counts += (
        np.abs(permuted_correlations)
        >= observed_abs_correlations
    )

# The +1 correction prevents zero empirical p-values under a finite null.
empirical_p_values = (
    exceedance_counts + 1
) / (
    N_PERMUTATIONS + 1
)

In [22]:
# =============================================================================
# Apply multiple-testing correction to primary comparisons
# =============================================================================

empirical_significance = pd.DataFrame(
    [
        {
            "tumor_rna_axis": tumor_axis,
            "cell_line_program": cell_program,
            "empirical_p_value": empirical_p_values[i, j],
        }
        for i, tumor_axis in enumerate(tumor_rna_axes)
        for j, cell_program in enumerate(cell_line_program_ids)
    ]
)

# The 100 frozen tumor–cell-line comparisons constitute one primary
# multiplicity family; no candidate-specific correction is applied.
empirical_significance["empirical_q_value"] = multipletests(
    empirical_significance["empirical_p_value"],
    method="fdr_bh",
)[1]

empirical_significance["primary_fdr_supported"] = (
    empirical_significance["empirical_q_value"] < FDR_THRESHOLD
)

primary_correlations = primary_correlations.merge(
    empirical_significance,
    on=["tumor_rna_axis", "cell_line_program"],
    how="left",
)

In [23]:
# =============================================================================
# Identify reciprocal nearest-neighbor matches
# =============================================================================

primary_correlations["tumor_nearest_neighbor"] = (
    primary_correlations["abs_pearson_r"]
    == primary_correlations
    .groupby("tumor_rna_axis")["abs_pearson_r"]
    .transform("max")
)

primary_correlations["cell_line_nearest_neighbor"] = (
    primary_correlations["abs_pearson_r"]
    == primary_correlations
    .groupby("cell_line_program")["abs_pearson_r"]
    .transform("max")
)

# Supported correspondence will later require reciprocity rather than
# forcing a global one-to-one assignment across potentially related programs.
primary_correlations["mutual_nearest_neighbor"] = (
    primary_correlations["tumor_nearest_neighbor"]
    & primary_correlations["cell_line_nearest_neighbor"]
)

### Secondary structural-support criteria

Secondary structural evidence is evaluated independently of the primary Pearson gate and is not used to redefine component orientation.

A secondary view is considered supportive only under the following prespecified rules:

- **Rank-based support:** the Pearson-oriented Spearman correlation must be positive and significant under an empirical gene-label permutation null with Benjamini–Hochberg FDR < 0.05 across the 100 frozen comparisons.
- **High-loading support:** the signed top-10% loading-overlap statistic must exceed its empirical gene-label permutation null with Benjamini–Hochberg FDR < 0.05 across the 100 frozen comparisons.
- The top-5% loading overlap remains a sensitivity analysis only and cannot independently establish secondary support.

The same frozen shared-gene universe, permutation count, random seed, and multiple-testing family size used for the primary comparison are retained.

For permutation-based secondary metrics, ICA orientation is recalculated within each permutation from the corresponding permuted Pearson correlation. This prevents the observed orientation from artificially favoring concordant signed evidence under the null.

`SUPPORTED_CORRESPONDENCE` therefore requires:

1. primary empirical Pearson FDR < 0.05;
2. mutual nearest-neighbor status;
3. support from at least one prespecified secondary structural view.

Secondary evidence cannot rescue a comparison that fails the primary Pearson gate.

In [24]:
# =============================================================================
# Prepare rank-normalized loading matrices for Spearman permutation testing
# =============================================================================

tumor_rank_matrix = np.array(
    tumor_shared_loadings.rank(axis=0, method="average"),
    dtype=float,
    copy=True,
)

cell_line_rank_matrix = np.array(
    cell_line_shared_loadings.rank(axis=0, method="average"),
    dtype=float,
    copy=True,
)

# Spearman correlation is Pearson correlation applied to ranks.
# Centering and unit-norm scaling allow all 100 correlations to be
# evaluated efficiently by matrix multiplication during permutation.
tumor_rank_matrix -= tumor_rank_matrix.mean(axis=0, keepdims=True)
tumor_rank_matrix /= np.linalg.norm(
    tumor_rank_matrix,
    axis=0,
    keepdims=True,
)

cell_line_rank_matrix -= cell_line_rank_matrix.mean(axis=0, keepdims=True)
cell_line_rank_matrix /= np.linalg.norm(
    cell_line_rank_matrix,
    axis=0,
    keepdims=True,
)

In [25]:
# =============================================================================
# Compute empirical oriented-Spearman permutation p-values
# =============================================================================

observed_oriented_spearman = (
    primary_correlations
    .pivot(
        index="tumor_rna_axis",
        columns="cell_line_program",
        values="oriented_spearman_r",
    )
    .reindex(
        index=tumor_rna_axes,
        columns=cell_line_program_ids,
    )
    .to_numpy()
)

rng = np.random.default_rng(RANDOM_SEED)
spearman_exceedance_counts = np.zeros_like(
    observed_oriented_spearman,
    dtype=int,
)

for _ in range(N_PERMUTATIONS):
    permuted_indices = rng.permutation(shared_gene_count)

    # Orientation is recalculated from the permuted primary Pearson
    # correlation so the observed sign convention is not imposed on the null.
    permuted_pearson = (
        tumor_loading_matrix.T
        @ cell_line_loading_matrix[permuted_indices, :]
    )
    permuted_orientation = np.sign(permuted_pearson)

    permuted_spearman = (
        tumor_rank_matrix.T
        @ cell_line_rank_matrix[permuted_indices, :]
    )
    permuted_oriented_spearman = (
        permuted_spearman * permuted_orientation
    )

    spearman_exceedance_counts += (
        permuted_oriented_spearman
        >= observed_oriented_spearman
    )

# Secondary rank support is directional after Pearson-based orientation:
# positive concordance, rather than absolute rank correlation, is the target.
spearman_empirical_p_values = (
    spearman_exceedance_counts + 1
) / (
    N_PERMUTATIONS + 1
)

In [26]:
# =============================================================================
# Apply multiple-testing correction to oriented Spearman support
# =============================================================================

spearman_significance = pd.DataFrame(
    [
        {
            "tumor_rna_axis": tumor_axis,
            "cell_line_program": cell_program,
            "spearman_empirical_p_value": spearman_empirical_p_values[i, j],
        }
        for i, tumor_axis in enumerate(tumor_rna_axes)
        for j, cell_program in enumerate(cell_line_program_ids)
    ]
)

# Rank-based secondary evidence forms its own prespecified multiplicity family
# across the same 100 frozen cross-system comparisons.
spearman_significance["spearman_empirical_q_value"] = multipletests(
    spearman_significance["spearman_empirical_p_value"],
    method="fdr_bh",
)[1]

primary_correlations = primary_correlations.merge(
    spearman_significance,
    on=["tumor_rna_axis", "cell_line_program"],
    how="left",
)

primary_correlations["rank_support"] = (
    (primary_correlations["oriented_spearman_r"] > 0)
    & (
        primary_correlations["spearman_empirical_q_value"]
        < FDR_THRESHOLD
    )
)

In [27]:
# =============================================================================
# Define observed signed high-loading overlap score
# =============================================================================

primary_correlations["concordant_top_count"] = (
    primary_correlations["concordant_positive_count"]
    + primary_correlations["concordant_negative_count"]
)

# The score rewards concordant shared extreme genes and penalizes discordant
# ones, while normalizing by the fixed top-loading set size.
primary_correlations["signed_top_loading_score"] = (
    primary_correlations["concordant_top_count"]
    - primary_correlations["discordant_count"]
) / primary_correlations["top_n"]

In [28]:
# =============================================================================
# Prepare signed top-loading matrices for permutation testing
# =============================================================================

tumor_raw_loading_matrix = np.array(
    tumor_shared_loadings,
    dtype=float,
    copy=True,
)

cell_line_raw_loading_matrix = np.array(
    cell_line_shared_loadings,
    dtype=float,
    copy=True,
)

top_n = int(
    np.ceil(
        len(tumor_shared_loadings)
        * TOP_LOADING_FRACTION
    )
)

tumor_top_mask = np.zeros(
    tumor_raw_loading_matrix.shape,
    dtype=bool,
)

cell_line_top_mask = np.zeros(
    cell_line_raw_loading_matrix.shape,
    dtype=bool,
)

for j in range(tumor_raw_loading_matrix.shape[1]):
    top_indices = np.argsort(
        np.abs(tumor_raw_loading_matrix[:, j])
    )[-top_n:]

    tumor_top_mask[top_indices, j] = True

for j in range(cell_line_raw_loading_matrix.shape[1]):
    top_indices = np.argsort(
        np.abs(cell_line_raw_loading_matrix[:, j])
    )[-top_n:]

    cell_line_top_mask[top_indices, j] = True

# Separate positive and negative extreme loadings using the same raw-loading
# definition as the observed signed high-loading overlap statistic.
tumor_top_positive = (
    tumor_top_mask
    & (tumor_raw_loading_matrix > 0)
).astype(int)

tumor_top_negative = (
    tumor_top_mask
    & (tumor_raw_loading_matrix < 0)
).astype(int)

cell_line_top_positive = (
    cell_line_top_mask
    & (cell_line_raw_loading_matrix > 0)
).astype(int)

cell_line_top_negative = (
    cell_line_top_mask
    & (cell_line_raw_loading_matrix < 0)
).astype(int)

In [29]:
# =============================================================================
# Compute empirical signed top-loading overlap permutation p-values
# =============================================================================

observed_top_loading_scores = (
    primary_correlations
    .pivot(
        index="tumor_rna_axis",
        columns="cell_line_program",
        values="signed_top_loading_score",
    )
    .reindex(
        index=tumor_rna_axes,
        columns=cell_line_program_ids,
    )
    .to_numpy()
)

rng = np.random.default_rng(RANDOM_SEED)
top_loading_exceedance_counts = np.zeros_like(
    observed_top_loading_scores,
    dtype=int,
)

for _ in range(N_PERMUTATIONS):
    permuted_indices = rng.permutation(shared_gene_count)

    # Recalculate ICA orientation from the permuted primary Pearson
    # correspondence rather than imposing the observed orientation.
    permuted_pearson = (
        tumor_loading_matrix.T
        @ cell_line_loading_matrix[permuted_indices, :]
    )
    permuted_orientation = np.sign(permuted_pearson)

    same_sign_overlap = (
        tumor_top_positive.T
        @ cell_line_top_positive[permuted_indices, :]
        + tumor_top_negative.T
        @ cell_line_top_negative[permuted_indices, :]
    )

    opposite_sign_overlap = (
        tumor_top_positive.T
        @ cell_line_top_negative[permuted_indices, :]
        + tumor_top_negative.T
        @ cell_line_top_positive[permuted_indices, :]
    )

    # Multiplication by the Pearson-derived orientation converts raw signed
    # overlap into concordance relative to the oriented cell-line component.
    permuted_top_loading_score = (
        (same_sign_overlap - opposite_sign_overlap)
        * permuted_orientation
        / top_n
    )

    top_loading_exceedance_counts += (
        permuted_top_loading_score
        >= observed_top_loading_scores
    )

top_loading_empirical_p_values = (
    top_loading_exceedance_counts + 1
) / (
    N_PERMUTATIONS + 1
)

In [30]:
# =============================================================================
# Apply multiple-testing correction to signed top-loading support
# =============================================================================

top_loading_significance = pd.DataFrame(
    [
        {
            "tumor_rna_axis": tumor_axis,
            "cell_line_program": cell_program,
            "top_loading_empirical_p_value": (
                top_loading_empirical_p_values[i, j]
            ),
        }
        for i, tumor_axis in enumerate(tumor_rna_axes)
        for j, cell_program in enumerate(cell_line_program_ids)
    ]
)

# Signed top-loading overlap constitutes its own prespecified secondary
# multiplicity family across the same 100 frozen comparisons.
top_loading_significance["top_loading_empirical_q_value"] = multipletests(
    top_loading_significance["top_loading_empirical_p_value"],
    method="fdr_bh",
)[1]

primary_correlations = primary_correlations.merge(
    top_loading_significance,
    on=["tumor_rna_axis", "cell_line_program"],
    how="left",
)

primary_correlations["top_loading_support"] = (
    (primary_correlations["signed_top_loading_score"] > 0)
    & (
        primary_correlations["top_loading_empirical_q_value"]
        < FDR_THRESHOLD
    )
)

In [31]:
# =============================================================================
# Combine prespecified secondary structural support
# =============================================================================

primary_correlations["secondary_structural_support"] = (
    primary_correlations["rank_support"]
    | primary_correlations["top_loading_support"]
)

# Keep the individual support routes explicit for interpretability rather
# than collapsing them into a single opaque multiview score.
primary_correlations["secondary_support_count"] = (
    primary_correlations[
        ["rank_support", "top_loading_support"]
    ]
    .sum(axis=1)
    .astype(int)
)

In [32]:
# =============================================================================
# Derive pair-level correspondence evidence flags
# =============================================================================

primary_correlations["supported_pair_candidate"] = (
    primary_correlations["primary_fdr_supported"]
    & primary_correlations["mutual_nearest_neighbor"]
    & primary_correlations["secondary_structural_support"]
)

# Primary-supported pairs that do not satisfy the complete reciprocal and
# multiview criterion remain eligible only for partial/ambiguous assessment.
primary_correlations["partial_pair_candidate"] = (
    primary_correlations["primary_fdr_supported"]
    & ~primary_correlations["supported_pair_candidate"]
)

primary_correlations["no_primary_support"] = (
    ~primary_correlations["primary_fdr_supported"]
)

In [33]:
# =============================================================================
# Summarize correspondence evidence by tumor RNA axis
# =============================================================================

tumor_axis_evidence = (
    primary_correlations
    .groupby("tumor_rna_axis", as_index=False)
    .agg(
        primary_supported_count=(
            "primary_fdr_supported",
            "sum",
        ),
        supported_pair_count=(
            "supported_pair_candidate",
            "sum",
        ),
        partial_pair_count=(
            "partial_pair_candidate",
            "sum",
        ),
        mutual_nearest_neighbor_count=(
            "mutual_nearest_neighbor",
            "sum",
        ),
    )
)

# Multiple primary-supported cell-line candidates indicate potential
# one-to-many structure or ambiguity and must not be silently forced to 1:1.
tumor_axis_evidence["multiple_primary_candidates"] = (
    tumor_axis_evidence["primary_supported_count"] > 1
)

In [34]:
# =============================================================================
# Identify best cell-line candidate for each tumor RNA axis
# =============================================================================

best_candidate_metrics = (
    primary_correlations
    .loc[primary_correlations["tumor_nearest_neighbor"]]
    [
        [
            "tumor_rna_axis",
            "cell_line_program",
            "pearson_r",
            "abs_pearson_r",
            "orientation_multiplier",
            "empirical_q_value",
            "oriented_spearman_r",
            "spearman_empirical_q_value",
            "signed_top_loading_score",
            "top_loading_empirical_q_value",
            "primary_fdr_supported",
            "mutual_nearest_neighbor",
            "rank_support",
            "top_loading_support",
            "secondary_structural_support",
            "supported_pair_candidate",
        ]
    ]
    .copy()
)

# Exact ties are retained rather than silently broken; they would represent
# genuine ambiguity in the primary structural nearest-neighbor criterion.
best_candidate_metrics["best_candidate_tie"] = (
    best_candidate_metrics
    .groupby("tumor_rna_axis")["cell_line_program"]
    .transform("size")
    .gt(1)
)

In [35]:
# =============================================================================
# Check exact nearest-neighbor ties
# =============================================================================

exact_ties = (
    best_candidate_metrics
    .loc[best_candidate_metrics["best_candidate_tie"]]
    [
        [
            "tumor_rna_axis",
            "cell_line_program",
            "abs_pearson_r",
        ]
    ]
    .copy()
)

print("Tumor RNA axes with exact best-match ties:", exact_ties["tumor_rna_axis"].nunique())

Tumor RNA axes with exact best-match ties: 0


In [36]:
# =============================================================================
# Build tumor-axis correspondence summary
# =============================================================================

best_candidate_metrics = (
    best_candidate_metrics
    .loc[~best_candidate_metrics["best_candidate_tie"]]
    .drop(columns="best_candidate_tie")
    .copy()
)

tumor_axis_correspondence = (
    tumor_axis_evidence
    .merge(
        best_candidate_metrics,
        on="tumor_rna_axis",
        how="left",
    )
)

# The selected row is the unique Pearson nearest neighbor for each tumor axis.
# Additional primary-supported candidates remain visible through the
# axis-level counts and are not discarded from the full pairwise table.

In [37]:
# =============================================================================
# Assign tumor-axis correspondence classes
# =============================================================================

conditions = [
    tumor_axis_correspondence["supported_pair_count"].eq(1),
    (
        tumor_axis_correspondence["primary_supported_count"].gt(1)
        & tumor_axis_correspondence["supported_pair_count"].eq(0)
    ),
    (
        tumor_axis_correspondence["primary_supported_count"].eq(1)
        & tumor_axis_correspondence["supported_pair_count"].eq(0)
    ),
    tumor_axis_correspondence["primary_supported_count"].eq(0),
]

choices = [
    "SUPPORTED_CORRESPONDENCE",
    "AMBIGUOUS_CORRESPONDENCE",
    "PARTIAL_CORRESPONDENCE",
    "NOT_RECOVERABLE",
]

# A fully supported reciprocal best match takes precedence even when additional
# primary-supported candidates exist; ambiguity is reserved for unresolved
# competition among multiple primary-supported candidates.
tumor_axis_correspondence["correspondence_class"] = np.select(
    conditions,
    choices,
    default="UNCLASSIFIED",
)

In [38]:
# =============================================================================
# Inspect tumor-axis correspondence classes
# =============================================================================

print(
    tumor_axis_correspondence["correspondence_class"]
    .value_counts()
)

tumor_axis_correspondence[
    [
        "tumor_rna_axis",
        "cell_line_program",
        "abs_pearson_r",
        "empirical_q_value",
        "mutual_nearest_neighbor",
        "rank_support",
        "top_loading_support",
        "primary_supported_count",
        "correspondence_class",
    ]
].sort_values(
    ["correspondence_class", "abs_pearson_r"],
    ascending=[True, False],
)

correspondence_class
AMBIGUOUS_CORRESPONDENCE    7
SUPPORTED_CORRESPONDENCE    3
Name: count, dtype: int64


,tumor_rna_axis,cell_line_program,abs_pearson_r,empirical_q_value,mutual_nearest_neighbor,rank_support,top_loading_support,primary_supported_count,correspondence_class
9,RNA_IC193,ICA_PROGRAM_33,0.281623,0.000222,False,True,True,6,AMBIGUOUS_CORRESPONDENCE
1,RNA_IC050,ICA_PROGRAM_33,0.252247,0.000222,False,True,True,9,AMBIGUOUS_CORRESPONDENCE
7,RNA_IC175,ICA_PROGRAM_13,0.216286,0.000222,False,True,True,8,AMBIGUOUS_CORRESPONDENCE
0,RNA_IC001,ICA_PROGRAM_33,0.163390,0.000222,False,True,False,8,AMBIGUOUS_CORRESPONDENCE
6,RNA_IC169,ICA_PROGRAM_13,0.161655,0.000222,False,True,False,3,AMBIGUOUS_CORRESPONDENCE
2,RNA_IC083,ICA_PROGRAM_07,0.136296,0.000222,False,True,False,4,AMBIGUOUS_CORRESPONDENCE
5,RNA_IC158,ICA_PROGRAM_46,0.105474,0.000222,False,True,False,5,AMBIGUOUS_CORRESPONDENCE
3,RNA_IC150,ICA_PROGRAM_09,0.554112,0.000222,True,True,True,7,SUPPORTED_CORRESPONDENCE
8,RNA_IC184,ICA_PROGRAM_13,0.428645,0.000222,True,True,True,8,SUPPORTED_CORRESPONDENCE
4,RNA_IC151,ICA_PROGRAM_29,0.275712,0.000222,True,True,True,7,SUPPORTED_CORRESPONDENCE


In [39]:
# =============================================================================
# Characterize nearest-neighbor competition
# =============================================================================

ranked_candidates = (
    primary_correlations
    .sort_values(
        ["tumor_rna_axis", "abs_pearson_r"],
        ascending=[True, False],
    )
    .assign(
        candidate_rank=lambda df: (
            df.groupby("tumor_rna_axis")
            .cumcount()
            .add(1)
        )
    )
)

top_two_candidates = (
    ranked_candidates
    .loc[ranked_candidates["candidate_rank"].le(2)]
    .pivot(
        index="tumor_rna_axis",
        columns="candidate_rank",
        values=["cell_line_program", "abs_pearson_r"],
    )
)

top_two_candidates.columns = [
    "best_cell_line_program",
    "second_cell_line_program",
    "best_abs_pearson_r",
    "second_abs_pearson_r",
]

top_two_candidates["nearest_neighbor_margin"] = (
    top_two_candidates["best_abs_pearson_r"]
    - top_two_candidates["second_abs_pearson_r"]
)

top_two_candidates.reset_index()

,tumor_rna_axis,best_cell_line_program,second_cell_line_program,best_abs_pearson_r,second_abs_pearson_r,nearest_neighbor_margin
0,RNA_IC001,ICA_PROGRAM_33,ICA_PROGRAM_06,0.16339,0.138162,0.025227
1,RNA_IC050,ICA_PROGRAM_33,ICA_PROGRAM_42,0.252247,0.208623,0.043624
2,RNA_IC083,ICA_PROGRAM_07,ICA_PROGRAM_20,0.136296,0.092494,0.043803
3,RNA_IC150,ICA_PROGRAM_09,ICA_PROGRAM_33,0.554112,0.477472,0.07664
4,RNA_IC151,ICA_PROGRAM_29,ICA_PROGRAM_06,0.275712,0.244391,0.031322
5,RNA_IC158,ICA_PROGRAM_46,ICA_PROGRAM_09,0.105474,0.085675,0.019799
6,RNA_IC169,ICA_PROGRAM_13,ICA_PROGRAM_20,0.161655,0.120875,0.040781
7,RNA_IC175,ICA_PROGRAM_13,ICA_PROGRAM_33,0.216286,0.184004,0.032281
8,RNA_IC184,ICA_PROGRAM_13,ICA_PROGRAM_09,0.428645,0.245381,0.183265
9,RNA_IC193,ICA_PROGRAM_33,ICA_PROGRAM_42,0.281623,0.243165,0.038458


In [40]:
# =============================================================================
# Attach nearest-neighbor competition metrics
# =============================================================================

tumor_axis_correspondence = (
    tumor_axis_correspondence
    .merge(
        top_two_candidates.reset_index(),
        on="tumor_rna_axis",
        how="left",
    )
)

# Nearest-neighbor margins characterize structural competition only.
# They are not used to modify the prespecified correspondence classes.

In [41]:
# =============================================================================
# Inspect tumor-arm metadata available for correspondence propagation
# =============================================================================

print(
    tumor_candidates.columns.tolist()
)

['rna_component', 'methylation_component', 'median_project_correlation', 'median_absolute_project_correlation', 'project_correlation_iqr', 'direction_consistency', 'valid_project_count', 'absolute_median_project_correlation', 'same_direction_ge_0_10_project_count', 'same_direction_ge_0_20_project_count', 'association_direction', 'magnitude_stratum', 'candidate_pair', 'matched_project_count', 'median_baseline', 'median_sex_adjusted', 'median_retained_fraction', 'median_absolute_correlation_change', 'sex_sensitivity_status', 'sex_sensitivity_reason', 'sex_sensitivity_analysis_scope', 'rna_nmf_factor', 'methylation_nmf_factor', 'rna_nmf_concordance', 'methylation_nmf_concordance', 'minimum_nmf_concordance', 'mean_nmf_concordance', 'nmf_analysis_scope', 'nmf_candidate_status', 'nmf_interpretation']


In [42]:
# =============================================================================
# Propagate transcriptomic correspondence to tumor cross-omic arms
# =============================================================================

tumor_arm_correspondence = (
    tumor_candidates
    .merge(
        tumor_axis_correspondence,
        left_on="rna_component",
        right_on="tumor_rna_axis",
        how="left",
        validate="many_to_one",
    )
    .rename(
        columns={
            "cell_line_program": "matched_cell_line_program",
        }
    )
)

# Tumor arms sharing the same RNA component inherit the same transcriptomic
# correspondence; they remain distinct cross-omic arms but do not constitute
# independent cross-system replication events.

In [43]:
# =============================================================================
# Inspect cell-line robustness metadata for evidence integration
# =============================================================================

print(
    cell_line_robustness.columns.tolist()
)

['program_id', 'discovery_rho', 'lineage_adjusted_rho', 'lineage_effect_retention', 'bootstrap_direction_preserved_fraction', 'near_primary_direction_preserved_fraction', 'single_lineage_influenced', 'lineage_coverage_adjusted_rho', 'coverage_effect_retention', 'individual_covariate_min_retention', 'individual_covariate_direction_fraction', 'joint_adjusted_rho', 'joint_covariate_effect_retention', 'lineage_sensitive', 'bootstrap_direction_unstable', 'near_primary_phenotype_sensitive', 'coverage_sensitive', 'individual_covariate_sensitive', 'joint_covariate_sensitive', 'association_unstable', 'unresolved_confounding', 'context_sensitive', 'robustness_category', 'lineage_eta_squared', 'most_influential_lineage', 'max_absolute_deviation', 'bootstrap_median', 'ci_lower', 'ci_upper', 'null_abs_95th_percentile', 'null_exceedance_fraction', 'cross_method_convergent', 'program_status']


In [44]:
# =============================================================================
# Attach cell-line robustness evidence
# =============================================================================

cell_line_robustness_handoff = cell_line_robustness[
    [
        "program_id",
        "robustness_category",
        "program_status",
        "lineage_eta_squared",
        "single_lineage_influenced",
        "coverage_sensitive",
        "association_unstable",
        "unresolved_confounding",
        "context_sensitive",
        "cross_method_convergent",
    ]
].copy()

tumor_arm_correspondence = (
    tumor_arm_correspondence
    .merge(
        cell_line_robustness_handoff,
        left_on="matched_cell_line_program",
        right_on="program_id",
        how="left",
        validate="many_to_one",
    )
    .drop(columns="program_id")
)

# Cell-line robustness qualifies the evidence associated with a structural
# match; it does not alter or upgrade the cross-system correspondence class.

In [45]:
# =============================================================================
# Inspect post-correction correspondence results
# =============================================================================

print(
    tumor_axis_correspondence["correspondence_class"]
    .value_counts()
)

tumor_axis_correspondence[
    [
        "tumor_rna_axis",
        "cell_line_program",
        "abs_pearson_r",
        "top_loading_support",
        "secondary_structural_support",
        "mutual_nearest_neighbor",
        "primary_supported_count",
        "correspondence_class",
    ]
].sort_values(
    ["correspondence_class", "abs_pearson_r"],
    ascending=[True, False],
)

correspondence_class
AMBIGUOUS_CORRESPONDENCE    7
SUPPORTED_CORRESPONDENCE    3
Name: count, dtype: int64


,tumor_rna_axis,cell_line_program,abs_pearson_r,top_loading_support,secondary_structural_support,mutual_nearest_neighbor,primary_supported_count,correspondence_class
9,RNA_IC193,ICA_PROGRAM_33,0.281623,True,True,False,6,AMBIGUOUS_CORRESPONDENCE
1,RNA_IC050,ICA_PROGRAM_33,0.252247,True,True,False,9,AMBIGUOUS_CORRESPONDENCE
7,RNA_IC175,ICA_PROGRAM_13,0.216286,True,True,False,8,AMBIGUOUS_CORRESPONDENCE
0,RNA_IC001,ICA_PROGRAM_33,0.163390,False,True,False,8,AMBIGUOUS_CORRESPONDENCE
6,RNA_IC169,ICA_PROGRAM_13,0.161655,False,True,False,3,AMBIGUOUS_CORRESPONDENCE
2,RNA_IC083,ICA_PROGRAM_07,0.136296,False,True,False,4,AMBIGUOUS_CORRESPONDENCE
5,RNA_IC158,ICA_PROGRAM_46,0.105474,False,True,False,5,AMBIGUOUS_CORRESPONDENCE
3,RNA_IC150,ICA_PROGRAM_09,0.554112,True,True,True,7,SUPPORTED_CORRESPONDENCE
8,RNA_IC184,ICA_PROGRAM_13,0.428645,True,True,True,8,SUPPORTED_CORRESPONDENCE
4,RNA_IC151,ICA_PROGRAM_29,0.275712,True,True,True,7,SUPPORTED_CORRESPONDENCE


In [46]:
# =============================================================================
# Inspect tumor-arm cross-system correspondence handoff
# =============================================================================

tumor_arm_correspondence[
    [
        "candidate_pair",
        "rna_component",
        "methylation_component",
        "matched_cell_line_program",
        "correspondence_class",
        "abs_pearson_r",
        "robustness_category",
        "program_status",
    ]
].sort_values(
    ["correspondence_class", "candidate_pair"]
)

,candidate_pair,rna_component,methylation_component,matched_cell_line_program,correspondence_class,abs_pearson_r,robustness_category,program_status
0,CROSS_OMIC_PAIR_02,RNA_IC083,METH_IC232,ICA_PROGRAM_07,AMBIGUOUS_CORRESPONDENCE,0.136296,ROBUSTNESS_SUPPORTED_CANDIDATE,candidate_ica_specific
3,CROSS_OMIC_PAIR_05,RNA_IC169,METH_IC023,ICA_PROGRAM_13,AMBIGUOUS_CORRESPONDENCE,0.161655,ROBUSTNESS_SUPPORTED_CANDIDATE,candidate_ica_specific
4,CROSS_OMIC_PAIR_06,RNA_IC175,METH_IC013,ICA_PROGRAM_13,AMBIGUOUS_CORRESPONDENCE,0.216286,ROBUSTNESS_SUPPORTED_CANDIDATE,candidate_ica_specific
5,CROSS_OMIC_PAIR_07,RNA_IC050,METH_IC234,ICA_PROGRAM_33,AMBIGUOUS_CORRESPONDENCE,0.252247,ROBUSTNESS_SUPPORTED_CANDIDATE,candidate_with_cross_method_support
7,CROSS_OMIC_PAIR_09,RNA_IC001,METH_IC107,ICA_PROGRAM_33,AMBIGUOUS_CORRESPONDENCE,0.163390,ROBUSTNESS_SUPPORTED_CANDIDATE,candidate_with_cross_method_support
8,CROSS_OMIC_PAIR_10,RNA_IC001,METH_IC241,ICA_PROGRAM_33,AMBIGUOUS_CORRESPONDENCE,0.163390,ROBUSTNESS_SUPPORTED_CANDIDATE,candidate_with_cross_method_support
9,CROSS_OMIC_PAIR_11,RNA_IC193,METH_IC109,ICA_PROGRAM_33,AMBIGUOUS_CORRESPONDENCE,0.281623,ROBUSTNESS_SUPPORTED_CANDIDATE,candidate_with_cross_method_support
11,CROSS_OMIC_PAIR_13,RNA_IC193,METH_IC033,ICA_PROGRAM_33,AMBIGUOUS_CORRESPONDENCE,0.281623,ROBUSTNESS_SUPPORTED_CANDIDATE,candidate_with_cross_method_support
12,CROSS_OMIC_PAIR_14,RNA_IC158,METH_IC013,ICA_PROGRAM_46,AMBIGUOUS_CORRESPONDENCE,0.105474,ROBUSTNESS_SUPPORTED_CANDIDATE,candidate_ica_specific
1,CROSS_OMIC_PAIR_03,RNA_IC184,METH_IC169,ICA_PROGRAM_13,SUPPORTED_CORRESPONDENCE,0.428645,ROBUSTNESS_SUPPORTED_CANDIDATE,candidate_ica_specific


In [47]:
# =============================================================================
# Load tumor program robustness evidence
# =============================================================================

TUMOR_ROBUSTNESS_PATH = (
    TUMOR_PROGRAM_DIR
    / "tcga_cross_omic_candidate_program_robustness_evidence.csv"
)

tumor_robustness = pd.read_csv(
    TUMOR_ROBUSTNESS_PATH
)

In [48]:
# =============================================================================
# Inspect tumor robustness metadata for evidence integration
# =============================================================================

print(
    tumor_robustness.columns.tolist()
)

['candidate_pair', 'rna_component', 'methylation_component', 'median_project_correlation', 'direction_consistency', 'lopo_absolute_shift', 'lopo_direction_preserved_all', 'bootstrap_ci_lower', 'bootstrap_ci_upper', 'bootstrap_direction_preservation_fraction', 'confounder_maximum_absolute_shift', 'confounder_direction_preserved_all', 'plate_median_absolute_shift', 'plate_project_direction_preservation_fraction', 'rna_seed_median', 'rna_subsample_median', 'methylation_seed_median', 'methylation_subsample_median', 'minimum_nmf_concordance']


In [49]:
# =============================================================================
# Attach tumor robustness evidence
# =============================================================================

tumor_robustness_handoff = tumor_robustness[
    [
        "candidate_pair",
        "lopo_absolute_shift",
        "lopo_direction_preserved_all",
        "bootstrap_direction_preservation_fraction",
        "confounder_maximum_absolute_shift",
        "confounder_direction_preserved_all",
        "plate_median_absolute_shift",
        "plate_project_direction_preservation_fraction",
        "rna_seed_median",
        "rna_subsample_median",
        "methylation_seed_median",
        "methylation_subsample_median",
        "minimum_nmf_concordance",
    ]
].copy()

tumor_arm_correspondence = (
    tumor_arm_correspondence
    .merge(
        tumor_robustness_handoff,
        on="candidate_pair",
        how="left",
        validate="one_to_one",
    )
)

# Tumor robustness qualifies the upstream evidence for each cross-omic arm.
# It does not upgrade or rescue its cross-system correspondence class.

In [50]:
# =============================================================================
# Inspect supported cross-system tumor arms with robustness context
# =============================================================================

tumor_arm_correspondence.loc[
    tumor_arm_correspondence["correspondence_class"]
    .eq("SUPPORTED_CORRESPONDENCE"),
    [
        "candidate_pair",
        "rna_component",
        "methylation_component",
        "matched_cell_line_program",
        "abs_pearson_r",
        "robustness_category",
        "lopo_direction_preserved_all",
        "bootstrap_direction_preservation_fraction",
        "confounder_direction_preserved_all",
        "plate_median_absolute_shift",
        "plate_project_direction_preservation_fraction",
    ],
].sort_values("abs_pearson_r", ascending=False)

,candidate_pair,rna_component,methylation_component,matched_cell_line_program,abs_pearson_r,robustness_category,lopo_direction_preserved_all,bootstrap_direction_preservation_fraction,confounder_direction_preserved_all,plate_median_absolute_shift,plate_project_direction_preservation_fraction
2,CROSS_OMIC_PAIR_04,RNA_IC150,METH_IC128,ICA_PROGRAM_09,0.554112,CONTEXT_SENSITIVE_CANDIDATE,True,1.0,True,0.002370,0.962963
1,CROSS_OMIC_PAIR_03,RNA_IC184,METH_IC169,ICA_PROGRAM_13,0.428645,ROBUSTNESS_SUPPORTED_CANDIDATE,True,1.0,True,0.006108,1.000000
10,CROSS_OMIC_PAIR_12,RNA_IC184,METH_IC128,ICA_PROGRAM_13,0.428645,ROBUSTNESS_SUPPORTED_CANDIDATE,True,1.0,True,0.018035,1.000000
6,CROSS_OMIC_PAIR_08,RNA_IC151,METH_IC050,ICA_PROGRAM_29,0.275712,ROBUSTNESS_SUPPORTED_CANDIDATE,True,1.0,True,0.006776,1.000000


In [51]:
# =============================================================================
# Annotate shared tumor RNA-axis structure
# =============================================================================

tumor_arm_correspondence["rna_axis_arm_count"] = (
    tumor_arm_correspondence
    .groupby("rna_component")["candidate_pair"]
    .transform("size")
)

tumor_arm_correspondence["shared_rna_axis"] = (
    tumor_arm_correspondence["rna_axis_arm_count"] > 1
)

# Cross-system correspondence is established at the unique tumor RNA-axis
# level. Multiple cross-omic arms sharing that axis remain distinct tumor
# programs but represent a single transcriptomic correspondence event.

In [52]:
# =============================================================================
# Build independent transcriptomic correspondence summary
# =============================================================================

tumor_axis_arm_structure = (
    tumor_arm_correspondence
    .groupby("rna_component", as_index=False)
    .agg(
        tumor_arm_count=("candidate_pair", "size"),
        tumor_candidate_pairs=(
            "candidate_pair",
            lambda values: ";".join(values),
        ),
    )
    .rename(
        columns={
            "rna_component": "tumor_rna_axis",
        }
    )
)

transcriptomic_event_summary = (
    tumor_axis_correspondence
    .merge(
        tumor_axis_arm_structure,
        on="tumor_rna_axis",
        how="left",
        validate="one_to_one",
    )
)

# Each row represents one independent tumor RNA-axis comparison event,
# irrespective of how many cross-omic tumor arms share that RNA component.

In [53]:
# =============================================================================
# Inspect independent transcriptomic correspondence events
# =============================================================================

transcriptomic_event_summary[
    [
        "tumor_rna_axis",
        "cell_line_program",
        "abs_pearson_r",
        "empirical_q_value",
        "mutual_nearest_neighbor",
        "rank_support",
        "top_loading_support",
        "primary_supported_count",
        "correspondence_class",
        "tumor_arm_count",
        "tumor_candidate_pairs",
    ]
].sort_values(
    ["correspondence_class", "abs_pearson_r"],
    ascending=[True, False],
)

,tumor_rna_axis,cell_line_program,abs_pearson_r,empirical_q_value,mutual_nearest_neighbor,rank_support,top_loading_support,primary_supported_count,correspondence_class,tumor_arm_count,tumor_candidate_pairs
9,RNA_IC193,ICA_PROGRAM_33,0.281623,0.000222,False,True,True,6,AMBIGUOUS_CORRESPONDENCE,2,CROSS_OMIC_PAIR_11;CROSS_OMIC_PAIR_13
1,RNA_IC050,ICA_PROGRAM_33,0.252247,0.000222,False,True,True,9,AMBIGUOUS_CORRESPONDENCE,1,CROSS_OMIC_PAIR_07
7,RNA_IC175,ICA_PROGRAM_13,0.216286,0.000222,False,True,True,8,AMBIGUOUS_CORRESPONDENCE,1,CROSS_OMIC_PAIR_06
0,RNA_IC001,ICA_PROGRAM_33,0.163390,0.000222,False,True,False,8,AMBIGUOUS_CORRESPONDENCE,2,CROSS_OMIC_PAIR_09;CROSS_OMIC_PAIR_10
6,RNA_IC169,ICA_PROGRAM_13,0.161655,0.000222,False,True,False,3,AMBIGUOUS_CORRESPONDENCE,1,CROSS_OMIC_PAIR_05
2,RNA_IC083,ICA_PROGRAM_07,0.136296,0.000222,False,True,False,4,AMBIGUOUS_CORRESPONDENCE,1,CROSS_OMIC_PAIR_02
5,RNA_IC158,ICA_PROGRAM_46,0.105474,0.000222,False,True,False,5,AMBIGUOUS_CORRESPONDENCE,1,CROSS_OMIC_PAIR_14
3,RNA_IC150,ICA_PROGRAM_09,0.554112,0.000222,True,True,True,7,SUPPORTED_CORRESPONDENCE,1,CROSS_OMIC_PAIR_04
8,RNA_IC184,ICA_PROGRAM_13,0.428645,0.000222,True,True,True,8,SUPPORTED_CORRESPONDENCE,2,CROSS_OMIC_PAIR_03;CROSS_OMIC_PAIR_12
4,RNA_IC151,ICA_PROGRAM_29,0.275712,0.000222,True,True,True,7,SUPPORTED_CORRESPONDENCE,1,CROSS_OMIC_PAIR_08


In [54]:
# =============================================================================
# Attach cell-line robustness to transcriptomic event summary
# =============================================================================

transcriptomic_event_summary = (
    transcriptomic_event_summary
    .merge(
        cell_line_robustness_handoff,
        left_on="cell_line_program",
        right_on="program_id",
        how="left",
        validate="many_to_one",
    )
    .drop(columns="program_id")
)

# Internal cell-line robustness contextualizes each independently derived
# correspondence event but does not modify its structural classification.

In [55]:
# =============================================================================
# Prepare pairwise cross-system matching artifact
# =============================================================================

pairwise_matching_metrics = primary_correlations[
    [
        "tumor_rna_axis",
        "cell_line_program",
        "pearson_r",
        "abs_pearson_r",
        "orientation_multiplier",
        "empirical_q_value",
        "primary_fdr_supported",
        "spearman_r",
        "oriented_spearman_r",
        "spearman_empirical_q_value",
        "rank_support",
        "signed_top_loading_score",
        "top_loading_empirical_q_value",
        "top_loading_support",
        "secondary_structural_support",
        "tumor_nearest_neighbor",
        "cell_line_nearest_neighbor",
        "mutual_nearest_neighbor",
        "supported_pair_candidate",
        "partial_pair_candidate",
        "no_primary_support",
    ]
].copy()

In [56]:
# =============================================================================
# Prepare transcriptomic correspondence summary artifact
# =============================================================================

correspondence_summary = transcriptomic_event_summary[
    [
        "tumor_rna_axis",
        "cell_line_program",
        "pearson_r",
        "abs_pearson_r",
        "orientation_multiplier",
        "empirical_q_value",
        "primary_supported_count",
        "mutual_nearest_neighbor",
        "rank_support",
        "top_loading_support",
        "correspondence_class",
        "best_cell_line_program",
        "second_cell_line_program",
        "best_abs_pearson_r",
        "second_abs_pearson_r",
        "nearest_neighbor_margin",
        "tumor_arm_count",
        "tumor_candidate_pairs",
        "robustness_category",
        "program_status",
        "context_sensitive",
        "association_unstable",
        "unresolved_confounding",
        "cross_method_convergent",
    ]
].copy()

In [57]:
# =============================================================================
# Resolve duplicated NMF concordance field
# =============================================================================

tumor_arm_correspondence = (
    tumor_arm_correspondence
    .rename(
        columns={
            "minimum_nmf_concordance_x": "minimum_nmf_concordance",
        }
    )
    .drop(columns="minimum_nmf_concordance_y")
)

# Both merged fields were identical across all 13 tumor arms, so one
# canonical representation is retained without loss of information.

In [58]:
# =============================================================================
# Prepare tumor-arm cross-system handoff artifact
# =============================================================================

tumor_arm_handoff = tumor_arm_correspondence[
    [
        "candidate_pair",
        "rna_component",
        "methylation_component",
        "median_project_correlation",
        "direction_consistency",
        "association_direction",
        "magnitude_stratum",
        "matched_cell_line_program",
        "pearson_r",
        "abs_pearson_r",
        "orientation_multiplier",
        "empirical_q_value",
        "mutual_nearest_neighbor",
        "rank_support",
        "top_loading_support",
        "correspondence_class",
        "rna_axis_arm_count",
        "shared_rna_axis",
        "robustness_category",
        "program_status",
        "context_sensitive",
        "association_unstable",
        "unresolved_confounding",
        "cross_method_convergent",
        "lopo_absolute_shift",
        "lopo_direction_preserved_all",
        "bootstrap_direction_preservation_fraction",
        "confounder_maximum_absolute_shift",
        "confounder_direction_preserved_all",
        "plate_median_absolute_shift",
        "plate_project_direction_preservation_fraction",
        "rna_seed_median",
        "rna_subsample_median",
        "methylation_seed_median",
        "methylation_subsample_median",
        "minimum_nmf_concordance",
    ]
].copy()

In [59]:
# =============================================================================
# Prepare shared-gene universe artifact
# =============================================================================

shared_gene_universe_artifact = shared_gene_universe[
    [
        "gene_symbol",
        "tumor_gene_id",
        "tumor_gene_name",
        "cell_line_gene_id",
    ]
].copy()

In [60]:
# =============================================================================
# Define cross-system comparison output paths
# =============================================================================

SHARED_GENE_UNIVERSE_OUTPUT_PATH = (
    OUTPUT_DIR
    / "400_cross_system_shared_gene_universe.csv"
)

PAIRWISE_MATCHING_OUTPUT_PATH = (
    OUTPUT_DIR
    / "400_cross_system_pairwise_matching_metrics.csv"
)

CORRESPONDENCE_SUMMARY_OUTPUT_PATH = (
    OUTPUT_DIR
    / "400_cross_system_correspondence_summary.csv"
)

TUMOR_ARM_HANDOFF_OUTPUT_PATH = (
    OUTPUT_DIR
    / "400_cross_system_tumor_arm_handoff.csv"
)

COMPARISON_METADATA_OUTPUT_PATH = (
    OUTPUT_DIR
    / "400_cross_system_comparison_metadata.json"
)

In [61]:
# =============================================================================
# Prepare cross-system comparison metadata
# =============================================================================

comparison_metadata = {
    "notebook": "400_cross_system_program_comparison",
    "analysis_scope": (
        "Transcriptomic structural correspondence between independently "
        "discovered TCGA and DepMap/GDSC candidate programs."
    ),
    "primary_unit": "unique_tumor_rna_axis",
    "tumor_candidate_arm_count": int(len(tumor_candidates)),
    "unique_tumor_rna_axis_count": int(len(tumor_rna_axes)),
    "cell_line_candidate_count": int(len(cell_line_program_ids)),
    "primary_comparison_count": int(len(pairwise_matching_metrics)),
    "shared_gene_count": int(len(shared_gene_universe_artifact)),
    "gene_harmonization": "unambiguous_shared_hgnc_symbol_intersection",
    "primary_metric": "pearson_loading_correlation",
    "matching_metric": "absolute_pearson_loading_correlation",
    "orientation_definition": "sign_of_pearson_loading_correlation",
    "secondary_metrics": [
        "pearson_oriented_spearman_loading_correlation",
        "signed_top_10_percent_loading_overlap",
    ],
    "top_loading_sensitivity_fraction": TOP_LOADING_SENSITIVITY_FRACTION,
    "permutation_count": N_PERMUTATIONS,
    "random_seed": RANDOM_SEED,
    "fdr_threshold": FDR_THRESHOLD,
    "multiplicity_scope": "100_prespecified_comparisons_per_metric_family",
    "supported_correspondence_rule": (
        "primary empirical FDR support plus mutual nearest-neighbor status "
        "plus at least one secondary structural support route"
    ),
    "correspondence_class_counts": (
        correspondence_summary["correspondence_class"]
        .value_counts()
        .to_dict()
    ),
    "interpretation_limitations": [
        (
            "Correspondence refers to the transcriptomic axis only and does "
            "not establish full epigenetic-transcriptomic reproduction."
        ),
        (
            "Internal robustness evidence from Phase 2 and Phase 3 "
            "contextualizes but does not determine correspondence classes."
        ),
        (
            "No pharmacogenomic phenotype information was used for "
            "cross-system matching, orientation, or thresholding."
        ),
        (
            "NOT_RECOVERABLE, if present, refers only to the frozen Phase 3 "
            "candidate space and not to absence from the cell-line transcriptome."
        ),
    ],
}

In [62]:
# =============================================================================
# Write cross-system comparison artifacts
# =============================================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

shared_gene_universe_artifact.to_csv(
    SHARED_GENE_UNIVERSE_OUTPUT_PATH,
    index=False,
)

pairwise_matching_metrics.to_csv(
    PAIRWISE_MATCHING_OUTPUT_PATH,
    index=False,
)

correspondence_summary.to_csv(
    CORRESPONDENCE_SUMMARY_OUTPUT_PATH,
    index=False,
)

tumor_arm_handoff.to_csv(
    TUMOR_ARM_HANDOFF_OUTPUT_PATH,
    index=False,
)

comparison_metadata["correspondence_class_counts"] = {
    key: int(value)
    for key, value in comparison_metadata[
        "correspondence_class_counts"
    ].items()
}

with COMPARISON_METADATA_OUTPUT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        comparison_metadata,
        file,
        indent=2,
    )

In [63]:
# =============================================================================
# Verify written cross-system comparison artifacts
# =============================================================================

written_artifacts_valid = (
    SHARED_GENE_UNIVERSE_OUTPUT_PATH.exists()
    and PAIRWISE_MATCHING_OUTPUT_PATH.exists()
    and CORRESPONDENCE_SUMMARY_OUTPUT_PATH.exists()
    and TUMOR_ARM_HANDOFF_OUTPUT_PATH.exists()
    and COMPARISON_METADATA_OUTPUT_PATH.exists()
    and len(pd.read_csv(SHARED_GENE_UNIVERSE_OUTPUT_PATH))
        == len(shared_gene_universe_artifact)
    and len(pd.read_csv(PAIRWISE_MATCHING_OUTPUT_PATH))
        == len(pairwise_matching_metrics)
    and len(pd.read_csv(CORRESPONDENCE_SUMMARY_OUTPUT_PATH))
        == len(correspondence_summary)
    and len(pd.read_csv(TUMOR_ARM_HANDOFF_OUTPUT_PATH))
        == len(tumor_arm_handoff)
)

with COMPARISON_METADATA_OUTPUT_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    written_metadata = json.load(file)

written_artifacts_valid = (
    written_artifacts_valid
    and written_metadata == comparison_metadata
)

print(
    "Cross-system comparison artifacts verified:",
    written_artifacts_valid,
)

Cross-system comparison artifacts verified: True


## Conclusion

Cross-system comparison was performed between 10 independently discovered
TCGA transcriptomic axes and 10 frozen DepMap/GDSC candidate programs,
yielding 100 prespecified structural comparisons across 2,389 shared genes.

Three independent transcriptomic correspondence events satisfied the
prespecified support criteria:

- `RNA_IC150` ↔ `ICA_PROGRAM_09`
- `RNA_IC184` ↔ `ICA_PROGRAM_13`
- `RNA_IC151` ↔ `ICA_PROGRAM_29`

The remaining seven tumor RNA axes were classified as
`AMBIGUOUS_CORRESPONDENCE`. No tumor RNA axis was classified as
`NOT_RECOVERABLE` within the frozen Phase 3 candidate space.

Propagation to the 13 TCGA cross-omic candidate arms yielded four arms with
supported transcriptomic correspondence because `CROSS_OMIC_PAIR_03` and
`CROSS_OMIC_PAIR_12` share the same `RNA_IC184` axis. These therefore represent
one, not two, independent cross-system transcriptomic correspondence events.

Internal robustness evidence from Phases 2 and 3 was retained as contextual
information and was not used to select, orient, rescue, or upgrade
cross-system matches. In particular, the `RNA_IC150` correspondence maps to
the context-sensitive cell-line candidate `ICA_PROGRAM_09`, which remains an
explicit qualification of that result.

These results support transcriptomic cross-system correspondence for a subset
of independently discovered candidate programs. They do not establish full
epigenetic-transcriptomic reproduction in cell lines, biological causality,
clinical resistance prediction, or therapeutic validity.

Notebook 400 produces cross-system comparison artifacts only. No consensus
programs are defined at this stage.